# AOU-1 — chr22 SMOKE variant. Phase M3 / Wave 1 validation. Fires `load_qc_cohort` with `interval_filter="chr22"` against the AoU v8 (or v9 post-migration) controlled-tier WGS MatrixTable. Emits 3 chr22-suffixed checkpointed MTs for live-Hail validation of m3-W1 Track 4 defensive assertions BEFORE any full-cohort rebuild fires.

**Purpose.** Validate that:
1. `_assert_checkpoint_nonempty(mt, uri, *, phase)` raises loudly on empty contents (or, in the happy case, returns silently with `count_rows() + count_cols() > 0`).
2. AOU-1 Cell 3.5 / 4.5 / 5.5 `gsutil du -s` assertions on `entries/entries/parts/` fire cleanly (with chr22-scaled threshold).
3. The auto-resume gate (`_validate_checkpoint_populated`) correctly distinguishes populated chr22 MTs from any stub-pattern outputs.
4. The underlying Hail / Dataproc image on the current platform (RW 2.0 or Legacy) produces populated chr22 MTs at all — if the catastrophe mechanism reproduces on chr22, we know the full-cohort rebuild will fail again and we pivot to 1000G AFR for Wave 2.

**Cost.** ~$30-75. ~3h cluster time on 16× n1-highmem-16 (256 vCPU minimum per `[[feedback_aou_cluster_sizing_for_ld_panel]]` — Hail's partition explosion is per-genome-region not per-output, so chr22 still needs the full cluster). 1/22nd of full-fire wall time × cluster cost.

**Prerequisite.** AOU-0 pre-check notebook must pass (`.planning/notebooks/AOU-0-precheck_template.ipynb`) — clone has Track 4 patches, env vars set, catastrophe inventory documented, hypothesis distinguisher recorded.

**Outputs.** 3 chr22-suffixed checkpointed MTs:
- `gs://${WORKSPACE_BUCKET}/ld/mt_afr_qc_chr22.mt`
- `gs://${WORKSPACE_BUCKET}/ld/mt_afr_pca_selfid_qc_chr22.mt`
- `gs://${WORKSPACE_BUCKET}/ld/mt_eur_qc_chr22.mt`

(Note: per `_intermediate_checkpoint_uri`, intermediate checkpoints will be chr22-suffixed too — `mt_*_post_split_chr22.mt`, `mt_*_post_sample_qc_chr22.mt`.)

**Smoke outputs are distinct from any future production fire** — `cohort_summary_m3_chr22.tsv` will be written instead of `cohort_summary_m3.tsv`.

**Cross-references:**
- `.planning/notebooks/AOU-1_template.ipynb` — the production AOU-1 (do NOT modify; this file is a smoke variant)
- `.planning/quick/260528-jvd-land-m3-w1-track-4-defensive-code-patche/260528-jvd-SUMMARY.md` — Track 4 patches
- `.planning/phases/m3-aou-afr-ld-panel-build/m3-CONTEXT.md` D-M3-10 — verification protocol
- `[[feedback_aou_dataproc_pyspark_submit_args]]` — Cell 1a PYSPARK_SUBMIT_ARGS lever
- `[[feedback_aou_cluster_sizing_for_ld_panel]]` — 256 vCPU minimum for v8 partition explosion


In [ ]:
# Cell 1a — Force Spark executor resources at the spark-submit boundary.
# CANONICAL PATTERN (per .planning/memory feedback_aou_dataproc_pyspark_submit_args
# baked 2026-05-12): on AoU's Dataproc + YARN cluster, hl.init(spark_conf=dict) is
# silently overridden by the cluster's spark-defaults.conf — the dict path doesn't
# beat YARN's executor-allocation policy. PYSPARK_SUBMIT_ARGS injected BEFORE any
# pyspark/hail import IS honored because it applies at the spark-submit boundary
# (highest Spark conf precedence).
#
# This cell MUST run before any other pyspark/hail import in the notebook.
# Pairs with naive_coalesce(2048) in aou_ld_panel.py:218 (DEC-2026-05-04-01
# v8 partition-explosion OOM remediation; anchor commit 8cc6f64).
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--conf spark.executor.cores=1 "
    "--conf spark.executor.memory=5g "
    "--conf spark.driver.cores=1 "
    "pyspark-shell"
)
print("PYSPARK_SUBMIT_ARGS set:", os.environ["PYSPARK_SUBMIT_ARGS"])

In [ ]:
# Cell 1b — Initialize Hail with spark_conf threading + verify executor.cores=1.
# Calls hl.init directly (NOT through init_hail wrapper) because the wrapper's
# spark_conf path is known broken on AoU YARN; the PYSPARK_SUBMIT_ARGS lever
# from Cell 1a is what actually binds the conf. The spark_conf dict here is
# belt-and-suspenders for portability — preserves the conf-by-dict path for
# environments where it works (local Spark, standalone clusters), while AoU
# binds via the env-var lever.
import sys
sys.path.insert(0, "/home/jupyter/coloc_analysis/src/python")
import hail as hl
hl.init(
    default_reference="GRCh38",
    log="/tmp/hail.log",
    quiet=True,
    spark_conf={
        "spark.executor.cores": "1",
        "spark.executor.memory": "5g",
        "spark.driver.cores": "1",
    },
)
# Pull cohort helpers AFTER Hail backend is up (so any aou_ld_panel-side
# Hail-dependent imports succeed):
from aou_ld_panel import load_qc_cohort, ANCESTRY_FIELD, KING_KINSHIP_THRESHOLD, _qc_checkpoint_uri

# Verify the patches are live (cores=1 confirms PYSPARK_SUBMIT_ARGS bound;
# _qc_checkpoint_uri import confirms commit 36e8062 is in the AoU clone):
sc_conf = hl.spark_context().getConf()
cores = sc_conf.get('spark.executor.cores')
assert cores == '1', (
    f"PYSPARK_SUBMIT_ARGS lever did not bind — got cores={cores}, expected '1'. "
    f"DO NOT proceed to Cell 3+ — the v8 partition-explosion OOM config is NOT live. "
    f"Action: Kernel menu → Restart Kernel; then re-fire Cell 1a + Cell 1b."
)
print("=== HAIL INIT ===")
print(f"  Hail version          : {hl.__version__}")
print(f"  spark.executor.cores  : {cores}  OK")
print(f"  spark.executor.memory : {sc_conf.get('spark.executor.memory')}")
print(f"  spark.driver.cores    : {sc_conf.get('spark.driver.cores')}")
print(f"  spark.master          : {sc_conf.get('spark.master')}")
print()
print("=== ENV ===")
print(f"  WORKSPACE_BUCKET = {os.environ['WORKSPACE_BUCKET']}")
print(f"  GOOGLE_PROJECT   = {os.environ['GOOGLE_PROJECT']}")
print(f"  WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH = {os.environ['WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH']}")
print()
print("=== PATCH VERIFICATION (commit 36e8062 — m3-W1-checkpoint-suffix; quick 260512-jd9) ===")
print(f"  AFR primary URI     : {_qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', False)}")
print(f"  AFR sensitivity URI : {_qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', True)}")
print(f"  EUR parity URI      : {_qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'eur', False)}")

In [ ]:
# Cell 3 — Primary AFR cohort (D-M3-07 PCA-primary) — chr22 SMOKE
mt_afr = load_qc_cohort(
    mt_path=os.environ["WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH"],
    ancestry="afr",
    sensitivity=False,
    interval_filter="chr22",  # chr22 smoke (DESIGN §3.5 path-isolated execution)
)
n_afr = mt_afr.count_cols()
n_var_afr = mt_afr.count_rows()
print(f"AFR PCA cohort (chr22): {n_afr} samples, {n_var_afr} variants")
# Already checkpointed to gs://${WORKSPACE_BUCKET}/ld/mt_afr_qc_chr22.mt by load_qc_cohort()


In [ ]:
# Cell 3.5 — Mandatory post-write bucket-contents validation (chr22 SMOKE variant).
# m3-W1-empty-mt-catastrophe (2026-05-21) regression guard, chr22-scaled.
#
# chr22 represents ~1/22 of the genome by variant count, so expected MT entries-
# dir size is ~500 MB - 2 GB for AFR primary, smaller for sensitivity cohorts.
# A 50 MB floor cleanly catches the catastrophe footer-stub state
# (~70 KiB total entries) without false-positiving on legitimately small
# chr22 cohorts. If a real chr22 fire reports < 50 MB entries, that's a
# signal worth investigating — it's well below typical chr22 cohort scale.
#
# Smoke-cell URI uses the chr22 interval-suffix path per
# _intermediate_checkpoint_uri / _qc_checkpoint_uri convention. Note that
# _qc_checkpoint_uri itself does NOT honor interval_filter (that's intermediate-
# only); we suffix manually to match the chr22 smoke output naming.
#
# Cross-references:
# - .planning/debug/m3-W1-empty-mt-catastrophe.md
# - [[feedback_aou_success_marker_not_evidence_of_data]]
# - [[feedback_hail_checkpoint_contract_violation]]
# - [[feedback_w1_catastrophe_hypothesis_distinguisher]]
#
# Cohort: Primary AFR chr22 smoke
import subprocess
_ckpt_uri = _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', False) + "_chr22"
_entries_dir = _ckpt_uri.rstrip('/') + '/entries/entries/parts/'
_r = subprocess.run(
    ['gsutil', 'du', '-s', _entries_dir],
    capture_output=True, text=True,
)
assert _r.returncode == 0, (
    f'bucket inspection failed at {_entries_dir}: returncode={_r.returncode}; '
    f'stderr={_r.stderr.strip()}. Likely entries/ directory absent — '
    f'the m3-W1 empty-MT catastrophe signature reproducing under chr22 smoke. '
    f'HALT — do not proceed to next cell. Notify Abby Doyle (Zendesk #57144) '
    f'and pivot Wave 2 to 1000G AFR substrate.'
)
_size_bytes = int(_r.stdout.split()[0])
_MIN_BYTES = 50000000  # 50 MB floor (chr22-scaled; production AOU-1 uses 1 GB)
assert _size_bytes > _MIN_BYTES, (
    f'MT entries at {_entries_dir} is {_size_bytes:,} bytes '
    f'(< {_MIN_BYTES:,} bytes chr22 floor) — empty-MT catastrophe regression '
    f'guard. chr22 cohort write produced a sub-floor MT; do NOT proceed. '
    f'See .planning/debug/m3-W1-empty-mt-catastrophe.md and '
    f'.planning/quick/260528-l8r-stage-aou-pre-check-chr22-smoke-aou-2-4-/MIGRATION-PLAYBOOK.md '
    f'failure-mode matrix.'
)
print(f'OK: {_ckpt_uri} populated ({_size_bytes / 10**6:.1f} MB at entries/entries/parts/)')


In [ ]:
# Cell 4 — AFR sensitivity cohort (D-M3-07 self-report Black/African American sensitivity) — chr22 SMOKE
mt_afr_selfid = load_qc_cohort(
    mt_path=os.environ["WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH"],
    ancestry="afr",
    sensitivity=True,
    interval_filter="chr22",
)
n_afr_selfid = mt_afr_selfid.count_cols()
print(f"AFR PCA + self-id Black/AA cohort (chr22): {n_afr_selfid} samples (subset of AFR PCA chr22 cohort)")
# Checkpoint at gs://${WORKSPACE_BUCKET}/ld/mt_afr_pca_selfid_qc_chr22.mt


In [ ]:
# Cell 4.5 — Mandatory post-write bucket-contents validation (chr22 SMOKE variant).
# m3-W1-empty-mt-catastrophe (2026-05-21) regression guard, chr22-scaled.
#
# chr22 represents ~1/22 of the genome by variant count, so expected MT entries-
# dir size is ~500 MB - 2 GB for AFR primary, smaller for sensitivity cohorts.
# A 50 MB floor cleanly catches the catastrophe footer-stub state
# (~70 KiB total entries) without false-positiving on legitimately small
# chr22 cohorts. If a real chr22 fire reports < 50 MB entries, that's a
# signal worth investigating — it's well below typical chr22 cohort scale.
#
# Smoke-cell URI uses the chr22 interval-suffix path per
# _intermediate_checkpoint_uri / _qc_checkpoint_uri convention. Note that
# _qc_checkpoint_uri itself does NOT honor interval_filter (that's intermediate-
# only); we suffix manually to match the chr22 smoke output naming.
#
# Cross-references:
# - .planning/debug/m3-W1-empty-mt-catastrophe.md
# - [[feedback_aou_success_marker_not_evidence_of_data]]
# - [[feedback_hail_checkpoint_contract_violation]]
# - [[feedback_w1_catastrophe_hypothesis_distinguisher]]
#
# Cohort: AFR sensitivity chr22 smoke
import subprocess
_ckpt_uri = _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', True) + "_chr22"
_entries_dir = _ckpt_uri.rstrip('/') + '/entries/entries/parts/'
_r = subprocess.run(
    ['gsutil', 'du', '-s', _entries_dir],
    capture_output=True, text=True,
)
assert _r.returncode == 0, (
    f'bucket inspection failed at {_entries_dir}: returncode={_r.returncode}; '
    f'stderr={_r.stderr.strip()}. Likely entries/ directory absent — '
    f'the m3-W1 empty-MT catastrophe signature reproducing under chr22 smoke. '
    f'HALT — do not proceed to next cell. Notify Abby Doyle (Zendesk #57144) '
    f'and pivot Wave 2 to 1000G AFR substrate.'
)
_size_bytes = int(_r.stdout.split()[0])
_MIN_BYTES = 50000000  # 50 MB floor (chr22-scaled; production AOU-1 uses 1 GB)
assert _size_bytes > _MIN_BYTES, (
    f'MT entries at {_entries_dir} is {_size_bytes:,} bytes '
    f'(< {_MIN_BYTES:,} bytes chr22 floor) — empty-MT catastrophe regression '
    f'guard. chr22 cohort write produced a sub-floor MT; do NOT proceed. '
    f'See .planning/debug/m3-W1-empty-mt-catastrophe.md and '
    f'.planning/quick/260528-l8r-stage-aou-pre-check-chr22-smoke-aou-2-4-/MIGRATION-PLAYBOOK.md '
    f'failure-mode matrix.'
)
print(f'OK: {_ckpt_uri} populated ({_size_bytes / 10**6:.1f} MB at entries/entries/parts/)')


In [ ]:
# Cell 5 — EUR parity cohort (D-M3-01) — chr22 SMOKE
mt_eur = load_qc_cohort(
    mt_path=os.environ["WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH"],
    ancestry="eur",
    sensitivity=False,
    interval_filter="chr22",
)
n_eur = mt_eur.count_cols()
print(f"EUR PCA cohort (chr22): {n_eur} samples")
# Checkpoint at gs://${WORKSPACE_BUCKET}/ld/mt_eur_qc_chr22.mt


In [ ]:
# Cell 5.5 — Mandatory post-write bucket-contents validation (chr22 SMOKE variant).
# m3-W1-empty-mt-catastrophe (2026-05-21) regression guard, chr22-scaled.
#
# chr22 represents ~1/22 of the genome by variant count, so expected MT entries-
# dir size is ~500 MB - 2 GB for AFR primary, smaller for sensitivity cohorts.
# A 50 MB floor cleanly catches the catastrophe footer-stub state
# (~70 KiB total entries) without false-positiving on legitimately small
# chr22 cohorts. If a real chr22 fire reports < 50 MB entries, that's a
# signal worth investigating — it's well below typical chr22 cohort scale.
#
# Smoke-cell URI uses the chr22 interval-suffix path per
# _intermediate_checkpoint_uri / _qc_checkpoint_uri convention. Note that
# _qc_checkpoint_uri itself does NOT honor interval_filter (that's intermediate-
# only); we suffix manually to match the chr22 smoke output naming.
#
# Cross-references:
# - .planning/debug/m3-W1-empty-mt-catastrophe.md
# - [[feedback_aou_success_marker_not_evidence_of_data]]
# - [[feedback_hail_checkpoint_contract_violation]]
# - [[feedback_w1_catastrophe_hypothesis_distinguisher]]
#
# Cohort: EUR parity chr22 smoke
import subprocess
_ckpt_uri = _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'eur', False) + "_chr22"
_entries_dir = _ckpt_uri.rstrip('/') + '/entries/entries/parts/'
_r = subprocess.run(
    ['gsutil', 'du', '-s', _entries_dir],
    capture_output=True, text=True,
)
assert _r.returncode == 0, (
    f'bucket inspection failed at {_entries_dir}: returncode={_r.returncode}; '
    f'stderr={_r.stderr.strip()}. Likely entries/ directory absent — '
    f'the m3-W1 empty-MT catastrophe signature reproducing under chr22 smoke. '
    f'HALT — do not proceed to next cell. Notify Abby Doyle (Zendesk #57144) '
    f'and pivot Wave 2 to 1000G AFR substrate.'
)
_size_bytes = int(_r.stdout.split()[0])
_MIN_BYTES = 50000000  # 50 MB floor (chr22-scaled; production AOU-1 uses 1 GB)
assert _size_bytes > _MIN_BYTES, (
    f'MT entries at {_entries_dir} is {_size_bytes:,} bytes '
    f'(< {_MIN_BYTES:,} bytes chr22 floor) — empty-MT catastrophe regression '
    f'guard. chr22 cohort write produced a sub-floor MT; do NOT proceed. '
    f'See .planning/debug/m3-W1-empty-mt-catastrophe.md and '
    f'.planning/quick/260528-l8r-stage-aou-pre-check-chr22-smoke-aou-2-4-/MIGRATION-PLAYBOOK.md '
    f'failure-mode matrix.'
)
print(f'OK: {_ckpt_uri} populated ({_size_bytes / 10**6:.1f} MB at entries/entries/parts/)')


In [ ]:
# Cell 6 — Disjoint-cohort sanity check (RESEARCH O5)
afr_samples = mt_afr.s.collect()
eur_samples = mt_eur.s.collect()
overlap = set(afr_samples) & set(eur_samples)
assert len(overlap) == 0, f"AFR and EUR cohorts overlap by {len(overlap)} samples; investigate!"
print(f"OK: AFR and EUR cohorts disjoint ({len(afr_samples)} + {len(eur_samples)} samples)")

In [ ]:
# Cell 7 — Cohort-summary table for the chr22 smoke validation
import pandas as pd
cohort_summary = pd.DataFrame({
    "cohort": ["AFR_pca_chr22", "AFR_pca_selfid_chr22", "EUR_pca_chr22"],
    "n_samples": [n_afr, n_afr_selfid, n_eur],
    "n_variants": [n_var_afr, mt_afr_selfid.count_rows(), mt_eur.count_rows()],
    "kinship_threshold": [KING_KINSHIP_THRESHOLD] * 3,
    "ancestry_field": [ANCESTRY_FIELD] * 3,
    "checkpoint_path": [
        _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', False) + '_chr22',
        _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', True) + '_chr22',
        _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'eur', False) + '_chr22',
    ],
    "interval_filter": ["chr22"] * 3,
})
cohort_summary.to_csv("cohort_summary_m3_chr22.tsv", sep="\t", index=False)
print(cohort_summary)
print()
print("chr22 smoke complete. If all 3 cohort-summary rows show non-zero n_samples AND n_variants,")
print("Track 4 defensive assertions are validated under live Hail; the full-cohort rebuild can proceed")
print("(assuming Track 1 credit recovery is also resolved). If any row shows 0, the catastrophe")
print("mechanism reproduces under the current platform — pivot Wave 2 to 1000G AFR.")


## chr22 smoke output: 3 chr22-suffixed checkpointed MTs in workspace bucket + `cohort_summary_m3_chr22.tsv` on env local disk.

Mirror `cohort_summary_m3_chr22.tsv` to NCSU GPFS at `.planning/phases/m3-aou-afr-ld-panel-build/cohort_summary_m3_chr22.tsv` after the fire (commit + push). The smoke validation memo lives at `.planning/quick/260528-l8r-stage-aou-pre-check-chr22-smoke-aou-2-4-/post-smoke-validation.md` (write this AFTER reviewing cohort_summary_m3_chr22.tsv).

**Next step if smoke passes:** Plan the full-cohort rebuild on the same Dataproc preset; expected cost ~$200-1500 depending on what Track 1 credit recovery and Abby Doyle's diagnostic yield. The Track 4 assertions in `load_qc_cohort` will catch any silent empty-MT recurrence.

**Next step if smoke fails:** Pivot Wave 2 to 1000G AFR substrate (free, NCSU-side, ~11 candidate-locus `.rds` files already at `data/processed/ld_reference/AFR/`). Document Wave 2 deviation in OSF amendment trail. Defer AoU AFR LD to grant-funded follow-on.